# 🏭 Taller de IA aplicada al mantenimiento industrial
**Notebook completo para Google Colab · sin APIs de pago · modelos locales con Ollama**

Este notebook resuelve los 5 puntos de la tarea. Cada parte tiene:

- 📘 una **sección de texto** que explica y responde la pregunta,
- 💻 **código comentado** línea por línea (qué hace cada bloque y por qué),
- ▶️ una **demostración** que puedes ejecutar.

| # | Punto de la tarea | Sección |
|---|---|---|
| 0 | Preparación del entorno (Ollama + modelos + utilidades) | Sección 0 |
| 1 | 4 *skills* propios con base en aihero.dev | Sección 1 |
| 2 | RAG industrial: chatbot con manuales técnicos de máquinas | Sección 2 |
| 3 | Chatbot con los documentos entregados en el curso | Sección 3 |
| 4 | Agente de mantenimiento (repuesto → almacén → orden de trabajo → historial) | Sección 4 |
| 5 | ¿Un ESP32 puede usar un LLM? Restricciones | Sección 5 |

### ⚙️ Cómo ejecutarlo
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4** (gratis). Sin GPU funciona, pero el LLM será muy lento.
2. Ejecuta las celdas **en orden** (o *Entorno de ejecución → Ejecutar todo*). La primera vez tarda ~10-15 min porque descarga el modelo (~5 GB).
3. Todo corre **dentro de Colab**: no se necesita ninguna API key ni se paga por uso.

> Si cambias de modelo, edita solo las variables `MODELO_LLM` y `MODELO_EMB` de la Sección 0.

<a name="0"></a>
# 0️⃣ Preparación del entorno

**Qué vamos a hacer aquí**

1. Verificar que Colab tiene GPU.
2. Instalar las librerías de Python.
3. Instalar **Ollama** (un servidor que ejecuta LLMs abiertos en tu propia máquina) y descargar dos modelos:
   - `qwen2.5:7b` → el **LLM** que conversa, escribe código y llama herramientas (*tool calling*).
   - `bge-m3` → un modelo de **embeddings** multilingüe (convierte texto en vectores para buscar por significado; sirve para preguntar en español sobre manuales en inglés).
4. Definir **utilidades comunes** que reutilizan todas las secciones: leer documentos, partirlos en fragmentos y llamar al LLM.

In [ ]:
# ── 0.1 Verificar la GPU ─────────────────────────────────────────────
# nvidia-smi lista la tarjeta gráfica. Si no aparece nada, el notebook sigue
# funcionando pero en CPU (mucho más lento).
import subprocess

salida = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True
).stdout.strip()
print("GPU detectada:", salida if salida else "NINGUNA (se usará CPU, será lento)")

In [ ]:
# ── 0.2 Instalar librerías de Python ─────────────────────────────────
# ollama      → cliente de Python para hablar con el servidor Ollama
# pypdf       → extraer texto de PDF
# python-docx → leer archivos Word (.docx)
# python-pptx → leer presentaciones (.pptx)
# pytest      → ejecutar las pruebas que genere el skill de implementación
# ipywidgets  → interfaz de chat dentro del notebook
!pip install -q ollama pypdf python-docx python-pptx pytest ipywidgets tqdm pandas matplotlib requests

In [ ]:
%%bash
# ── 0.3 Instalar Ollama ──────────────────────────────────────────────
# zstd, pciutils y lshw los necesita el instalador de Ollama para
# descomprimir el binario y detectar la GPU.
apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1
# Script oficial de instalación (requiere internet, que Colab sí tiene).
curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
ollama --version

In [ ]:
# ── 0.4 Levantar el servidor de Ollama en segundo plano ──────────────
import subprocess, time, requests

def iniciar_ollama():
    """Inicia 'ollama serve' si aún no está corriendo y espera a que responda."""
    url = "http://127.0.0.1:11434"
    try:                                   # ¿ya está activo?
        requests.get(url, timeout=2)
        print("✅ Ollama ya estaba corriendo")
        return
    except Exception:
        pass
    # Popen lanza el proceso sin bloquear la celda; guardamos su log en un archivo.
    subprocess.Popen(["ollama", "serve"],
                     stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(40):                    # espera hasta ~40 s
        try:
            requests.get(url, timeout=2)
            print("✅ Servidor Ollama listo")
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama no arrancó. Revisa el archivo ollama.log")

iniciar_ollama()

In [ ]:
# ── 0.5 Elegir y descargar los modelos ───────────────────────────────
# Cambia estos nombres si quieres otro modelo (consulta https://ollama.com/library).
#  - Poca VRAM / más rápido: "llama3.2:3b"  o  "qwen2.5:3b"
#  - Mejor calidad:          "qwen2.5:14b" (cabe en T4 cuantizado, pero es más lento)
MODELO_LLM = "qwen2.5:7b"   # conversación, código y tool calling
MODELO_EMB = "bge-m3"       # embeddings multilingües (≈1.2 GB)

# El prefijo ! ejecuta un comando de consola; {VAR} inserta variables de Python.
!ollama pull {MODELO_LLM}
!ollama pull {MODELO_EMB}

In [ ]:
# ── 0.6 Utilidades para llamar al LLM ────────────────────────────────
import ollama

def llm(mensajes, sistema=None, temperatura=0.2, **extra):
    """Llamada de bajo nivel al LLM.
    - mensajes: lista [{"role": "user"|"assistant", "content": "..."}]
    - sistema:  instrucciones de comportamiento (system prompt)
    - temperatura: 0 = determinista, 1 = más creativo
    - extra: parámetros adicionales (por ejemplo tools=[...])"""
    msgs = ([{"role": "system", "content": sistema}] if sistema else []) + list(mensajes)
    return ollama.chat(
        model=MODELO_LLM, messages=msgs,
        options={"temperature": temperatura, "num_ctx": 8192},  # num_ctx = memoria de contexto en tokens
        **extra
    )

def llm_texto(texto, sistema=None, temperatura=0.2):
    """Atajo: envía un solo mensaje de usuario y devuelve solo el texto de respuesta."""
    return llm([{"role": "user", "content": texto}], sistema, temperatura).message.content

# Prueba rápida de que todo funciona
print(llm_texto("Responde solo con la palabra: listo"))

In [ ]:
# ── 0.7 Utilidades para leer y fragmentar documentos ─────────────────
from pathlib import Path
import numpy as np

EXT_SOPORTADAS = {".pdf", ".docx", ".pptx", ".txt", ".md", ".csv"}

def leer_documento(ruta):
    """Lee un archivo y devuelve una lista de tuplas (número_de_página, texto).
    Para formatos sin páginas (docx, txt) devuelve una sola 'página' 1."""
    ruta = Path(ruta)
    ext = ruta.suffix.lower()
    if ext == ".pdf":
        from pypdf import PdfReader
        lector = PdfReader(str(ruta))
        return [(i + 1, pagina.extract_text() or "") for i, pagina in enumerate(lector.pages)]
    if ext == ".docx":
        import docx
        d = docx.Document(str(ruta))
        texto = "\n".join(p.text for p in d.paragraphs)
        for tabla in d.tables:                       # también capturamos las tablas
            for fila in tabla.rows:
                texto += "\n" + " | ".join(c.text for c in fila.cells)
        return [(1, texto)]
    if ext == ".pptx":
        from pptx import Presentation
        prs = Presentation(str(ruta))
        return [(i, "\n".join(sh.text_frame.text for sh in s.shapes if sh.has_text_frame))
                for i, s in enumerate(prs.slides, 1)]      # una "página" por diapositiva
    if ext in {".txt", ".md", ".csv"}:
        return [(1, ruta.read_text(encoding="utf-8", errors="ignore"))]
    raise ValueError(f"Formato no soportado: {ext}")

def dividir_en_fragmentos(paginas, fuente, tam=1000, solape=200):
    """Parte el texto en 'chunks' de ~tam caracteres con un solape entre ellos.
    - ¿Por qué fragmentar? Un LLM tiene contexto limitado y la búsqueda es más precisa
      sobre trozos pequeños que sobre un manual de 100 páginas.
    - ¿Por qué solape? Para que una idea cortada entre dos trozos aparezca completa en uno.
    Cada fragmento guarda su fuente y página para poder CITARLA en la respuesta."""
    fragmentos = []
    for pagina, texto in paginas:
        texto = " ".join(texto.split())               # normaliza espacios y saltos de línea
        inicio = 0
        while inicio < len(texto):
            fin = min(inicio + tam, len(texto))
            if fin < len(texto):                       # intenta cortar en un espacio, no a media palabra
                corte = texto.rfind(" ", inicio, fin)
                if corte > inicio + tam // 2:
                    fin = corte
            trozo = texto[inicio:fin]
            if len(trozo.strip()) > 50:                # descarta trozos casi vacíos
                fragmentos.append({"fuente": fuente, "pagina": pagina, "texto": trozo})
            if fin >= len(texto):
                break
            inicio = max(fin - solape, inicio + 1)
    return fragmentos

# Prueba: un texto corto genera un solo fragmento
print(dividir_en_fragmentos([(1, "Lubricar los rodamientos cada 2000 horas. " * 5)], "demo.txt")[:1])

In [ ]:
# ── 0.8 Índice vectorial (la "memoria" de búsqueda del RAG) ──────────
from tqdm.auto import tqdm

class IndiceRAG:
    """Guarda fragmentos de texto + sus embeddings y permite buscar por similitud.

    Idea clave: un *embedding* es una lista de números (vector) que representa el
    SIGNIFICADO de un texto. Textos con significado parecido → vectores parecidos.
    Buscamos comparando el vector de la pregunta con el de cada fragmento
    (similitud coseno) y nos quedamos con los k más cercanos."""

    def __init__(self, nombre):
        self.nombre = nombre
        self.fragmentos = []      # lista de dicts {fuente, pagina, texto}
        self.matriz = None        # matriz numpy (n_fragmentos × dimensión)

    def _embeber(self, textos, lote=16):
        """Convierte textos en vectores usando Ollama, por lotes para ir más rápido."""
        vectores = []
        for i in range(0, len(textos), lote):
            r = ollama.embed(model=MODELO_EMB, input=textos[i:i + lote])
            vectores += r.embeddings
        m = np.array(vectores, dtype="float32")
        m /= np.linalg.norm(m, axis=1, keepdims=True) + 1e-9   # normalizar → coseno = producto punto
        return m

    def agregar_documentos(self, rutas):
        """Lee, fragmenta y vectoriza los documentos indicados."""
        nuevos = []
        for ruta in rutas:
            paginas = leer_documento(ruta)
            frs = dividir_en_fragmentos(paginas, Path(ruta).name)
            print(f"  📄 {Path(ruta).name}: {len(paginas)} páginas → {len(frs)} fragmentos")
            nuevos += frs
        # vectorizamos todos los fragmentos nuevos (con barra de progreso)
        vecs = []
        for i in tqdm(range(0, len(nuevos), 16), desc="Embeddings"):
            vecs.append(self._embeber([f["texto"] for f in nuevos[i:i + 16]]))
        vecs = np.vstack(vecs)
        self.fragmentos += nuevos
        self.matriz = vecs if self.matriz is None else np.vstack([self.matriz, vecs])
        print(f"✅ Índice '{self.nombre}': {len(self.fragmentos)} fragmentos en total")

    def buscar(self, pregunta, k=5):
        """Devuelve los k fragmentos más parecidos a la pregunta."""
        q = self._embeber([pregunta])[0]
        puntajes = self.matriz @ q                    # similitud coseno con todos a la vez
        mejores = np.argsort(-puntajes)[:k]           # índices de los k mayores
        return [{**self.fragmentos[i], "puntaje": float(puntajes[i])} for i in mejores]

def formatear_contexto(fragmentos):
    """Junta los fragmentos recuperados en un bloque de texto con su fuente."""
    return "\n\n".join(f"[{f['fuente']}, pág. {f['pagina']}]\n{f['texto']}" for f in fragmentos)

In [ ]:
# ── 0.9 Interfaz de chat simple dentro del notebook ──────────────────
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

def crear_chat(funcion_respuesta, titulo="Chatbot"):
    """Muestra una cajita de chat. 'funcion_respuesta' recibe la pregunta (str)
    y devuelve la respuesta (str en Markdown)."""
    salida = widgets.Output(layout={"border": "1px solid #ccc", "height": "380px", "overflow_y": "auto"})
    caja = widgets.Text(placeholder="Escribe tu pregunta y pulsa Enter…", layout=widgets.Layout(width="80%"))
    boton = widgets.Button(description="Enviar", button_style="primary")

    def enviar(_):
        pregunta = caja.value.strip()
        if not pregunta:
            return
        caja.value = ""
        with salida:
            display(Markdown(f"**🧑 Tú:** {pregunta}"))
            display(Markdown(f"**🤖 Bot:** {funcion_respuesta(pregunta)}"))

    boton.on_click(enviar)
    caja.on_submit(enviar)
    display(widgets.HTML(f"<h4>{titulo}</h4>"), salida, widgets.HBox([caja, boton]))

<a name="1"></a>
# 1️⃣ Skills propios (inspirados en aihero.dev)

## 📘 Respuesta: ¿qué es un *skill* y cómo lo diseñamos?

En AI Hero (Matt Pocock) un **skill** es una capacidad "empaquetada": un archivo Markdown (`SKILL.md`) con **instrucciones y recursos para hacer bien una sola tarea**. Los puntos clave que aplicamos:

- **Skill ≠ tool.** Una *tool* es una función que el agente **llama** (buscar en BD, enviar correo). Un *skill* son **instrucciones que el agente lee** (cómo redactar un ticket, cómo hacer TDD).
- **Divulgación progresiva (*progressive disclosure*).** El agente solo ve el **nombre y la descripción** de cada skill; el contenido completo se carga en el contexto **únicamente cuando la tarea lo requiere**. Así no se gastan tokens en cada turno (a diferencia de meter todo en un `AGENTS.md`).
- **Formato:** carpeta `skills/<nombre>/SKILL.md` con un encabezado YAML (`name`, `description`) y el cuerpo con el proceso paso a paso.
- **Enfoque "ingeniería real, no *vibe coding*":** los skills del catálogo (`grill-me`, `to-spec`, `to-tickets`, `tdd`, `implement`…) fuerzan disciplina: alinear antes de construir, convertir conversaciones en tickets, probar primero.

### Los 4 skills que definimos

| Skill | Para qué sirve | Se inspira en |
|---|---|---|
| `documentar-ticket` | Convierte una petición informal en un ticket completo (descripción del trabajo por hacer, alcance, criterios de aceptación…) | `to-tickets` |
| `implementar-con-pruebas` | Genera código **y pruebas** siguiendo TDD (rojo → verde → refactor) | `tdd`, `implement` |
| `explicar-en-html` | Genera una página HTML autocontenida que explica en detalle el trabajo realizado | los reportes HTML del catálogo |
| `capturar-requerimientos` | Lee documentos provistos y extrae requerimientos trazables, ambigüedades y preguntas | `grill-with-docs`, `to-spec` |

Después de crearlos programamos un **cargador de skills** que replica la divulgación progresiva y probamos cada uno con el LLM local.

In [ ]:
# ── 1.1 Crear la carpeta de skills y escribir los 4 archivos SKILL.md ─
# Cada skill es un archivo Markdown con "frontmatter" (entre ---) y un cuerpo.
# El frontmatter es lo ÚNICO que el agente ve al inicio (nombre + descripción);
# el cuerpo se carga solo si el skill se elige.
CARPETA_SKILLS = Path("skills")

SKILLS = {}

# ───────── Skill 1: documentar un ticket de trabajo ─────────
SKILLS["documentar-ticket"] = """---
name: documentar-ticket
description: Convierte una solicitud informal de trabajo en un ticket documentado (descripción, alcance, criterios de aceptación, tareas y riesgos). Úsalo cuando alguien describa un trabajo por hacer y haya que dejarlo registrado antes de implementarlo.
---
# Documentar un ticket de trabajo

## Cuándo usarlo
- Llega una petición de trabajo (correo, chat, llamada) y aún no hay ticket.
- Se necesita describir con precisión QUÉ se va a hacer, sin mezclarlo con el CÓMO.

## Proceso
1. Lee toda la solicitud. Identifica: quién lo pide, qué sistema/equipo afecta, resultado esperado y fecha.
2. NO inventes datos. Todo lo que falte va en "Preguntas abiertas".
3. Redacta el título en imperativo y máximo 10 palabras (ej. "Agregar consulta de almacén al agente").
4. Escribe la descripción del trabajo por hacer en lenguaje claro: problema → resultado esperado.
5. Define alcance: qué SÍ incluye y qué NO incluye (evita que el ticket crezca).
6. Escribe criterios de aceptación verificables (cada uno debe poder responderse con sí/no).
7. Divide el trabajo en tareas pequeñas (máx. 1 día cada una), ordenadas.
8. Lista riesgos y dependencias.

## Formato de salida (Markdown, respeta este orden)
# [TKT-0001] Título
**Tipo:** funcionalidad | corrección | mejora | investigación
**Solicitante:** ...   **Prioridad:** alta | media | baja   **Fecha objetivo:** ...

## Descripción del trabajo
(2-5 frases: problema y resultado esperado)

## Alcance
- Incluye: ...
- No incluye: ...

## Criterios de aceptación
- [ ] ...

## Tareas
1. ...

## Riesgos y dependencias
- ...

## Preguntas abiertas
- ...

## Reglas
- Escribe en español.
- No incluyas código ni decisiones de implementación en el ticket.
- Si no se indica número de ticket usa TKT-0001.
"""

# ───────── Skill 2: implementación, código y pruebas ─────────
SKILLS["implementar-con-pruebas"] = """---
name: implementar-con-pruebas
description: Implementa una función o módulo en Python con pruebas automáticas usando TDD (rojo, verde, refactor). Úsalo cuando se pida generar código y sus pruebas a partir de una especificación.
---
# Implementación con pruebas (TDD)

## Proceso
1. Entiende la especificación. Lista mentalmente entradas, salidas, casos borde y errores posibles.
2. ROJO: escribe primero las pruebas con pytest (casos normales, casos borde, entradas inválidas que deben lanzar excepción).
3. VERDE: escribe el código MÍNIMO que hace pasar todas las pruebas.
4. REFACTOR: mejora nombres, agrega docstring y type hints sin cambiar el comportamiento.

## Reglas del código
- Solo biblioteca estándar de Python.
- Funciones pequeñas, con docstring y type hints.
- Valida las entradas y lanza ValueError con mensaje claro.
- Usa EXACTAMENTE las fórmulas que da la especificación; no las cambies.
- Las pruebas importan el código así: from solucion import nombre_de_la_funcion
- Para comparar decimales usa pytest.approx.

## Formato de salida (OBLIGATORIO)
Responde con EXACTAMENTE dos bloques de código Python, en este orden, y cada uno
empieza con su comentario de archivo:

```python
# archivo: solucion.py
...
```

```python
# archivo: test_solucion.py
...
```

Después de los bloques escribe una sección breve "Notas" (máximo 5 líneas) con
supuestos y casos cubiertos. No agregues nada más.
"""

# ───────── Skill 3: explicación en HTML ─────────
SKILLS["explicar-en-html"] = """---
name: explicar-en-html
description: Genera una página HTML autocontenida que explica en detalle el trabajo realizado (qué se pidió, qué se hizo, cómo funciona, cómo probarlo). Úsalo al terminar una tarea para documentarla de forma visual.
---
# Explicar el trabajo realizado en HTML

## Proceso
1. Lee el material recibido (ticket, código, pruebas, resultados).
2. Escribe para una persona técnica que NO vio el trabajo: explica el porqué, no solo el qué.
3. Muestra fragmentos de código cortos y comentados, no el código completo.

## Secciones obligatorias (en este orden)
1. Resumen ejecutivo (3-4 frases)
2. Qué se pidió
3. Qué se hizo (lista de cambios)
4. Cómo funciona (paso a paso, con fragmentos de código)
5. Decisiones y alternativas descartadas
6. Cómo probarlo (comandos y resultado esperado)
7. Limitaciones y próximos pasos

## Reglas técnicas del HTML
- UN solo archivo, con el CSS dentro de <style>. Sin recursos externos (nada de CDN, fuentes ni imágenes remotas).
- Empieza con <!DOCTYPE html>, <html lang="es"> y <meta name="viewport" ...>.
- Diseño limpio, legible y responsive (max-width ~ 900px, tipografía del sistema).
- Bloques de código dentro de <pre><code>, con fondo oscuro.
- Escapa los caracteres < > & dentro del código.

## Salida
Devuelve SOLO el HTML completo. Sin explicaciones antes o después y sin cercos de código (```).
"""

# ───────── Skill 4: capturar requerimientos de documentos ─────────
SKILLS["capturar-requerimientos"] = """---
name: capturar-requerimientos
description: Lee documentos provistos (especificaciones, actas, correos, manuales) y captura requerimientos trazables, ambigüedades y preguntas abiertas. Úsalo cuando haya que extraer qué debe hacer un sistema a partir de texto existente.
---
# Capturar requerimientos desde documentos

## Proceso
1. Lee el documento completo antes de extraer nada.
2. Extrae solo lo que el documento dice o implica claramente. NO inventes requerimientos.
3. Clasifica cada uno:
   - RF = funcional (qué hace el sistema)
   - RNF = no funcional (rendimiento, seguridad, disponibilidad, usabilidad)
   - REST = restricción (tecnología, normativa, presupuesto, plazo)
   - SUP = supuesto
4. Redacta cada requerimiento como una frase verificable: "El sistema debe ..." (con un valor medible si existe).
5. Registra la fuente (sección, página o cita corta de máximo 12 palabras).
6. Señala ambigüedades (palabras como "rápido", "fácil", "adecuado") y contradicciones.
7. Formula preguntas para quien escribió el documento.

## Formato de salida (Markdown)
## Requerimientos
| ID | Tipo | Requerimiento | Prioridad | Fuente | Cómo se verifica |
|---|---|---|---|---|---|
| RF-01 | RF | El sistema debe ... | Alta/Media/Baja/No especificada | sección o cita | ... |

## Ambigüedades y contradicciones
- ...

## Preguntas abiertas
- ...

## Reglas
- Escribe en español.
- Si el documento no da prioridad, escribe "No especificada"; no la supongas.
- Una fila = un solo requerimiento (no mezcles dos en la misma frase).
"""

# Escribimos cada skill en su carpeta: skills/<nombre>/SKILL.md
for nombre, contenido in SKILLS.items():
    carpeta = CARPETA_SKILLS / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    (carpeta / "SKILL.md").write_text(contenido, encoding="utf-8")

print("Skills creados:")
for p in sorted(CARPETA_SKILLS.glob("*/SKILL.md")):
    print(" •", p)

In [ ]:
# ── 1.2 Cargador de skills con "divulgación progresiva" ──────────────
import re

def leer_skill(nombre):
    """Lee un SKILL.md y separa (metadatos, cuerpo)."""
    texto = (CARPETA_SKILLS / nombre / "SKILL.md").read_text(encoding="utf-8")
    m = re.match(r"---\n(.*?)\n---\n(.*)", texto, re.S)          # frontmatter + cuerpo
    meta = dict(linea.split(": ", 1) for linea in m.group(1).splitlines() if ": " in linea)
    return meta, m.group(2).strip()

def catalogo():
    """Devuelve SOLO nombre + descripción de cada skill (lo único que el agente ve al inicio)."""
    return {p.parent.name: leer_skill(p.parent.name)[0]["description"]
            for p in sorted(CARPETA_SKILLS.glob("*/SKILL.md"))}

def elegir_skill(tarea):
    """El LLM decide qué skill usar mirando únicamente el catálogo (no el cuerpo).
    Es el 'puntero de contexto': el cuerpo del skill se carga después, solo si se elige."""
    lista = "\n".join(f"- {n}: {d}" for n, d in catalogo().items())
    respuesta = llm_texto(
        f"Catálogo de skills:\n{lista}\n\nTarea del usuario:\n{tarea}\n\n"
        "Responde SOLO con el nombre exacto del skill más adecuado.",
        temperatura=0)
    for nombre in catalogo():
        if nombre in respuesta:
            return nombre
    return None

def ejecutar_skill(nombre, entrada, temperatura=0.2):
    """Carga el cuerpo del skill como instrucciones (system prompt) y le pasa la entrada."""
    _, cuerpo = leer_skill(nombre)
    return llm_texto(entrada, sistema=cuerpo, temperatura=temperatura)

print("Catálogo visible para el agente (solo nombre y descripción):\n")
for n, d in catalogo().items():
    print(f"• {n}\n    {d}\n")

### 🎯 Demo del skill 1 — `documentar-ticket`
Primero dejamos que el LLM **elija el skill** leyendo solo el catálogo, y luego lo ejecutamos con una petición informal.

In [ ]:
SOLICITUD = """Hola, necesitamos que el agente de mantenimiento de la planta pueda, cuando un
operador reporte una falla en un equipo, buscar el repuesto, ver si hay en el almacén,
abrir la orden de trabajo y dejar anotada la falla en el historial. Lo pide el jefe de
mantenimiento (Carlos Ruiz) y lo necesitamos para fin de trimestre."""

skill_elegido = elegir_skill("Documenta este trabajo como ticket: " + SOLICITUD)
print("🧭 Skill elegido por el LLM:", skill_elegido)

ticket = ejecutar_skill("documentar-ticket", SOLICITUD)
display(Markdown(ticket))

### 🎯 Demo del skill 2 — `implementar-con-pruebas`
Este demo cierra el ciclo de TDD de forma automática: el LLM genera **código + pruebas**, las guardamos en archivos, **ejecutamos pytest** y, si fallan, le devolvemos el error al LLM para que corrija (hasta 3 intentos). Eso es lo que distingue un skill de simplemente "pedir código".

In [ ]:
import sys, os

def implementar(tarea, max_intentos=3, carpeta="trabajo_impl"):
    """Genera código + pruebas con el skill y las valida con pytest, reintentando si fallan."""
    carpeta = Path(carpeta)
    carpeta.mkdir(exist_ok=True)
    mensaje = tarea
    respuesta = ""
    for intento in range(1, max_intentos + 1):
        print(f"\n🔁 Intento {intento}/{max_intentos}")
        respuesta = ejecutar_skill("implementar-con-pruebas", mensaje)

        # Extraemos los bloques ```python ... ``` y los asignamos por su comentario "# archivo:"
        bloques = re.findall(r"```python\n(.*?)```", respuesta, re.S)
        codigo = next((b for b in bloques if "# archivo: solucion.py" in b), None)
        pruebas = next((b for b in bloques if "# archivo: test_solucion.py" in b), None)
        if not (codigo and pruebas):
            print("⚠️ Formato incorrecto, pido que lo corrija")
            mensaje = tarea + "\n\nTu respuesta anterior no respetó el formato de dos bloques con '# archivo:'. Repite siguiendo el formato."
            continue

        (carpeta / "solucion.py").write_text(codigo, encoding="utf-8")
        (carpeta / "test_solucion.py").write_text(pruebas, encoding="utf-8")

        # Ejecutamos pytest como proceso aparte (sin caché para que siempre use el archivo nuevo)
        r = subprocess.run(
            [sys.executable, "-m", "pytest", "-q", "--no-header", "-p", "no:cacheprovider", str(carpeta)],
            capture_output=True, text=True, env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
        print(r.stdout[-800:])
        if r.returncode == 0:
            print("✅ Todas las pruebas pasan")
            return respuesta, True

        # Realimentación: le mostramos el fallo al LLM (esto es el ciclo rojo → verde)
        mensaje = (f"{tarea}\n\nTu intento anterior falló estas pruebas:\n{r.stdout[-1500:]}\n\n"
                   f"Código anterior:\n{codigo}\n\nCorrige el código (o las pruebas si están mal) y respeta el formato.")
    print("❌ No se logró pasar las pruebas en el máximo de intentos")
    return respuesta, False

TAREA_IMPL = """Implementa la función calcular_indicadores(horas_calendario, horas_paro, num_fallas)
para mantenimiento industrial. Debe devolver un diccionario con:
- 'mtbf' = (horas_calendario - horas_paro) / num_fallas   (tiempo medio entre fallas, en horas)
- 'mttr' = horas_paro / num_fallas                         (tiempo medio de reparación, en horas)
- 'disponibilidad' = mtbf / (mtbf + mttr) * 100            (en porcentaje)
Valida: horas_calendario > 0, 0 <= horas_paro <= horas_calendario, num_fallas >= 1."""

respuesta_impl, ok = implementar(TAREA_IMPL)
display(Markdown(respuesta_impl))

### 🎯 Demo del skill 3 — `explicar-en-html`
Le pasamos al skill el **ticket** (demo 1) y el **código + pruebas** (demo 2). Devuelve una página HTML que guardamos como `explicacion.html` y mostramos dentro del notebook.

In [ ]:
import html as htmllib

# Reunimos el material que el skill debe explicar
codigo_final = Path("trabajo_impl/solucion.py").read_text() if Path("trabajo_impl/solucion.py").exists() else "(sin código)"
pruebas_final = Path("trabajo_impl/test_solucion.py").read_text() if Path("trabajo_impl/test_solucion.py").exists() else "(sin pruebas)"

material = f"""TICKET:
{ticket}

CÓDIGO IMPLEMENTADO (solucion.py):
{codigo_final}

PRUEBAS (test_solucion.py):
{pruebas_final}

RESULTADO DE PYTEST: {"todas las pruebas pasaron" if ok else "hubo pruebas fallidas"}"""

respuesta_html = ejecutar_skill("explicar-en-html", material, temperatura=0.3)

# Por si el modelo agregó texto extra, recortamos desde <!DOCTYPE html> hasta </html>
m = re.search(r"<!DOCTYPE html>.*</html>", respuesta_html, re.S | re.I)
pagina = m.group(0) if m else respuesta_html
Path("explicacion.html").write_text(pagina, encoding="utf-8")
print("💾 Guardado en explicacion.html")

# Mostramos la página dentro de un iframe para que su CSS no afecte al notebook
display(HTML(f'<iframe srcdoc="{htmllib.escape(pagina)}" width="100%" height="650" style="border:1px solid #ccc"></iframe>'))

### 🎯 Demo del skill 4 — `capturar-requerimientos`
Creamos un documento de ejemplo con requerimientos **escritos de forma natural** (con ambigüedades a propósito), lo leemos con `leer_documento()` y dejamos que el skill extraiga los requerimientos. Puedes cambiar `RUTA_DOC` por tu propio PDF/Word.

In [ ]:
# Documento de ejemplo (sustitúyelo por tu propio archivo para probar con datos reales)
Path("especificacion_ejemplo.txt").write_text("""ESPECIFICACIÓN - SISTEMA DE GESTIÓN DE MANTENIMIENTO (borrador)

1. Alcance. El sistema apoyará al equipo de mantenimiento de la Planta 2 (bombas, motores y compresores).

2. Reporte de fallas. El operador podrá reportar una falla indicando el equipo y una descripción.
El sistema debería responder rápido y ser fácil de usar desde una tableta en planta.

3. Repuestos. Al reportar una falla, el sistema sugerirá los repuestos necesarios y consultará
el stock del almacén. Si no hay stock, avisará al jefe de compras.

4. Órdenes de trabajo. Cada falla generará una orden de trabajo con prioridad. Las fallas que
detengan la línea de producción son prioridad máxima.

5. Historial. Todas las fallas se registrarán por equipo. Los reportes deben estar disponibles
siempre que sea necesario.

6. Restricciones. Debe funcionar sin conexión a internet externa por política de seguridad de la planta.
El presupuesto de licencias es cero. Entrega estimada: marzo.
""", encoding="utf-8")

RUTA_DOC = "especificacion_ejemplo.txt"
texto_doc = "\n".join(t for _, t in leer_documento(RUTA_DOC))   # unimos todas las páginas

requerimientos = ejecutar_skill("capturar-requerimientos", f"DOCUMENTO ({RUTA_DOC}):\n\n{texto_doc}")
display(Markdown(requerimientos))

### ✅ Conclusión de la Sección 1
- Los 4 skills quedaron guardados en `skills/<nombre>/SKILL.md` (puedes usarlos tal cual en **Claude Code** copiándolos a `~/.claude/skills/`).
- Cada skill define **cuándo usarlo, el proceso, el formato de salida y las reglas**, y por eso los resultados son repetibles.
- El cargador demuestra la idea central de aihero.dev: el agente solo conoce **nombre + descripción** y carga el cuerpo **bajo demanda**.
- Los skills se pueden **encadenar**: `capturar-requerimientos` → `documentar-ticket` → `implementar-con-pruebas` → `explicar-en-html`.

<a name="2"></a>
# 2️⃣ RAG industrial: chatbot con manuales técnicos de máquinas

## 📘 Respuesta: ¿qué es un RAG y cómo se arma?

**RAG** (*Retrieval-Augmented Generation*) = el LLM **no responde de memoria**, sino con base en fragmentos de **tus** documentos que se recuperan primero. Es ideal para mantenimiento: los manuales son extensos, específicos de cada máquina y el modelo no los conoce.

```
 ┌──────────── INGESTA (una vez) ────────────┐      ┌─────────── CONSULTA (cada pregunta) ───────────┐
 │ PDF manuales → texto → fragmentos (chunks)│      │ pregunta → embedding → top-k fragmentos        │
 │      → embeddings (bge-m3) → índice       │      │   → prompt con contexto → LLM (Ollama) → respuesta│
 └───────────────────────────────────────────┘      │   con citas [archivo, página]                  │
                                                    └────────────────────────────────────────────────┘
```

**Decisiones de diseño**
- **Ollama** corre el LLM y los embeddings localmente (sin pagar API, y los manuales no salen de la planta).
- **bge-m3** es multilingüe: el técnico pregunta en español y los manuales están en inglés.
- El prompt obliga al modelo a **citar la fuente** y a decir "no encuentro" si la información no está (reduce alucinaciones, clave en seguridad industrial).
- Guardamos **fuente + página** de cada fragmento para que el técnico pueda verificar en el manual original.

### Paso 1 — Descargar manuales
Se intentan descargar manuales públicos de bombas centrífugas (Xylem/Bell & Gossett, Pentair/Hypro) y una guía de mantenimiento de bombas del LANL. Si alguna URL deja de funcionar, el notebook usa un **manual de ejemplo** para que todo siga corriendo. También puedes subir tus propios PDF a la carpeta `manuales/`.

In [ ]:
# ── 2.1 Descargar manuales técnicos ──────────────────────────────────
CARPETA_MANUALES = Path("manuales")
CARPETA_MANUALES.mkdir(exist_ok=True)

# nombre local → URL pública del manual (puedes agregar más líneas)
MANUALES = {
    "bomba_series80_xylem.pdf": "https://www.xylem.com/siteassets/brand/bell-amp-gossett/resources/manual/series-80-in-line-mounted-centrifugal-pump-p81629g.pdf",
    "bomba_hypro_9303.pdf":     "https://www.flowandfiltrationsolutions.pentair.com/content/dam/extranet/web/nam/hypro/manuals/l-1594-9303-series.pdf",
    "bomba_hypro_9313.pdf":     "https://pentair.com/content/dam/extranet/web/nam/hypro/manuals/l-1588-9313-centrifugal-pumps-forcefield-iom.pdf",
    "guia_mantenimiento_bombas_lanl.pdf": "https://engstandards.lanl.gov/om/pdfs/Approved/412_R1.pdf",
}

def descargar_pdf(nombre, url):
    """Descarga un PDF y verifica que realmente sea un PDF (algunos sitios devuelven una página HTML de error)."""
    destino = CARPETA_MANUALES / nombre
    if destino.exists():
        return True
    try:
        r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200 and r.content[:4] == b"%PDF":
            destino.write_bytes(r.content)
            print(f"✅ {nombre} ({len(r.content)/1e6:.1f} MB)")
            return True
        print(f"⚠️ {nombre}: respuesta no válida (código {r.status_code})")
    except Exception as e:
        print(f"⚠️ {nombre}: no se pudo descargar ({type(e).__name__})")
    return False

for nombre, url in MANUALES.items():
    descargar_pdf(nombre, url)

# Plan B: manual FICTICIO de ejemplo, para que el notebook funcione aunque no haya descargas
if not list(CARPETA_MANUALES.glob("*.pdf")):
    print("\n❗ No se descargó ningún PDF. Creando un manual de ejemplo (ficticio).")
    (CARPETA_MANUALES / "manual_ejemplo_bomba.txt").write_text("""
MANUAL DE OPERACIÓN Y MANTENIMIENTO - BOMBA CENTRÍFUGA BC-100 (documento de ejemplo)

1. SEGURIDAD. Antes de cualquier intervención aplique bloqueo y etiquetado (LOTO): desconecte la energía
eléctrica, cierre las válvulas de succión y descarga y alivie la presión de la carcasa. Use guantes y gafas.

2. ARRANQUE. Verifique que la carcasa esté cebada (llena de líquido). Nunca opere la bomba en seco: el sello
mecánico se daña en segundos. Abra la válvula de succión, arranque con la válvula de descarga casi cerrada y ábrala
gradualmente. Compruebe el sentido de giro del motor antes de acoplar.

3. LUBRICACIÓN. Los rodamientos se lubrican con grasa a base de litio NLGI 2 cada 2000 horas de operación o cada
3 meses, lo que ocurra primero. No sobrelubrique: el exceso de grasa eleva la temperatura. Temperatura máxima del
rodamiento: 80 °C.

4. SELLO MECÁNICO. Una fuga continua de más de 10 gotas por minuto indica desgaste del sello. Reemplace el sello
completo (código SM-BP101) y revise el estado del eje y la camisa.

5. VIBRACIÓN. Valores superiores a 7.1 mm/s RMS indican desalineación, desbalance o rodamientos dañados. Revise el
acoplamiento, la alineación láser (tolerancia 0.05 mm) y el estado de los rodamientos.

6. CAVITACIÓN. Ruido similar a grava, caída de caudal y vibración indican cavitación. Verifique el nivel de
succión, filtros obstruidos y que el NPSH disponible sea mayor que el requerido.

7. PLAN DE MANTENIMIENTO. Diario: inspección visual de fugas y ruido. Mensual: lectura de vibración y temperatura.
Semestral: alineación y cambio de grasa. Anual: inspección del impulsor y cambio del sello.
""", encoding="utf-8")

print("\nArchivos disponibles en manuales/:", [p.name for p in CARPETA_MANUALES.iterdir()])

In [ ]:
# ── 2.2 Construir el índice (extraer → fragmentar → embeddings) ──────
archivos_manuales = [p for p in sorted(CARPETA_MANUALES.iterdir()) if p.suffix.lower() in EXT_SOPORTADAS]

indice_manuales = IndiceRAG("manuales-planta")
indice_manuales.agregar_documentos(archivos_manuales)    # tarda unos minutos la primera vez

In [ ]:
# ── 2.3 Probar solo la RECUPERACIÓN (sin LLM) ────────────────────────
# Es buena práctica revisar qué fragmentos trae el buscador antes de pedirle al LLM
# que responda: si el contexto es malo, la respuesta será mala.
pregunta_test = "¿Cada cuánto tiempo hay que lubricar los rodamientos?"
for f in indice_manuales.buscar(pregunta_test, k=3):
    print(f"[{f['fuente']} | pág. {f['pagina']} | similitud {f['puntaje']:.2f}]")
    print(f["texto"][:300], "...\n")

In [ ]:
# ── 2.4 Generación: prompt + LLM ─────────────────────────────────────
PROMPT_RAG = """Eres un asistente de mantenimiento industrial. Respondes preguntas de técnicos
usando SOLO los FRAGMENTOS de manuales que se te entregan.

Reglas:
1. Si la respuesta no está en los fragmentos, responde exactamente: "No encuentro esa información en los manuales cargados."
2. Cita la fuente de cada dato al final de la frase, con el formato [archivo, pág. N].
3. Si hay riesgos de seguridad (energía, presión, partes giratorias, temperatura), menciónalos primero.
4. Responde en español, de forma clara; usa pasos numerados cuando sea un procedimiento.
5. No inventes valores numéricos (torques, tolerancias, intervalos) que no estén en los fragmentos."""

def responder_manuales(pregunta, k=5):
    """Pipeline RAG completo: recuperar → armar prompt con contexto → generar."""
    fragmentos = indice_manuales.buscar(pregunta, k)                  # 1) RETRIEVAL
    prompt = f"FRAGMENTOS:\n{formatear_contexto(fragmentos)}\n\nPREGUNTA: {pregunta}"
    return llm_texto(prompt, sistema=PROMPT_RAG, temperatura=0.1), fragmentos   # 2) GENERATION

for pregunta in ["¿Qué debo hacer antes de arrancar la bomba por primera vez?",
                 "¿Qué precauciones de seguridad debo tomar antes de dar mantenimiento a la bomba?",
                 "¿Cuál es la capital de Francia?"]:        # esta última debe responder "No encuentro..."
    respuesta, frs = responder_manuales(pregunta)
    display(Markdown(f"**❓ {pregunta}**\n\n{respuesta}"))
    print("Fuentes consultadas:", sorted({(f['fuente'], f['pagina']) for f in frs}), "\n")

In [ ]:
# ── 2.5 Chatbot interactivo con los manuales ─────────────────────────
crear_chat(lambda p: responder_manuales(p)[0], "🤖 Chatbot de manuales de planta")

### ✅ Conclusión de la Sección 2
- Se implementó un **RAG 100 % local**: PDF → fragmentos → embeddings (`bge-m3`) → búsqueda coseno → LLM en Ollama con citas.
- **Limitaciones reales** de un RAG industrial: los PDF escaneados necesitan OCR; las tablas y diagramas se pierden con `pypdf`; y la búsqueda solo por vectores puede fallar con códigos exactos (mejora: búsqueda **híbrida** vectores + palabras clave BM25).
- **Seguridad:** el chatbot es una ayuda de consulta; las acciones críticas (torques, LOTO) deben verificarse siempre en el manual original — por eso el bot cita página.

<a name="3"></a>
# 3️⃣ Chatbot con los documentos entregados en el curso

## 📘 Qué cambia respecto a la Sección 2

Reutilizamos el mismo motor (`IndiceRAG`), pero un chatbot de estudio necesita algo más:

1. **Memoria de conversación:** el estudiante pregunta *"¿y cuál es su ventaja?"* y el bot debe saber a qué se refiere.
2. **Reformulación de la pregunta:** antes de buscar, el LLM reescribe la pregunta de seguimiento como una pregunta **independiente** (si no, el buscador recibiría "¿y su ventaja?" y no encontraría nada).
3. **Formatos variados:** los materiales de curso suelen ser PDF, Word y PowerPoint (`leer_documento` ya los soporta).
4. **Prompt de tutor:** explica con claridad, cita la diapositiva/página y admite cuándo el material no cubre el tema.

### Cómo cargar tus documentos del curso
- **Opción A:** pon `SUBIR_DOCS_CURSO = True` y selecciona los archivos desde tu computador.
- **Opción B:** monta Google Drive y copia los archivos a la carpeta `docs_curso/`.
- Si la carpeta está vacía, el notebook usa los manuales de la Sección 2 como sustituto, para que puedas ver el funcionamiento.

In [ ]:
# ── 3.1 Cargar los documentos del curso ──────────────────────────────
CARPETA_CURSO = Path("docs_curso")
CARPETA_CURSO.mkdir(exist_ok=True)

SUBIR_DOCS_CURSO = False     # ← cámbialo a True para abrir el selector de archivos

if SUBIR_DOCS_CURSO:
    from google.colab import files
    for nombre, datos in files.upload().items():           # abre el diálogo de subida
        (CARPETA_CURSO / nombre).write_bytes(datos)

# Opción B (Google Drive): descomenta y ajusta la ruta
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/MiCurso/* docs_curso/

docs_curso = [p for p in sorted(CARPETA_CURSO.iterdir()) if p.suffix.lower() in EXT_SOPORTADAS]
if not docs_curso:
    print("⚠️ docs_curso/ está vacía → uso los manuales de la Sección 2 como sustituto.")
    docs_curso = archivos_manuales

indice_curso = IndiceRAG("curso")
indice_curso.agregar_documentos(docs_curso)

In [ ]:
# ── 3.2 Chatbot conversacional con memoria ───────────────────────────
PROMPT_CURSO = """Eres un tutor del curso. Respondes SOLO con base en los FRAGMENTOS del material
entregado. Explica con claridad y, si ayuda, con un ejemplo breve. Cita la fuente al final
con el formato [archivo, pág. N]. Si el material no cubre la pregunta, dilo claramente y
no inventes. Responde en español."""

class ChatbotCurso:
    def __init__(self, indice, k=5, memoria=6):
        self.indice = indice
        self.k = k
        self.memoria = memoria          # cuántos mensajes recientes recordar
        self.historial = []             # lista de (rol, texto)

    def _reformular(self, pregunta):
        """Convierte una pregunta de seguimiento en una pregunta autónoma usando el historial."""
        if not self.historial:
            return pregunta
        conversacion = "\n".join(f"{rol}: {txt}" for rol, txt in self.historial[-self.memoria:])
        return llm_texto(
            f"Conversación:\n{conversacion}\n\nPregunta de seguimiento: {pregunta}\n\n"
            "Reescribe la pregunta de seguimiento como una pregunta completa e independiente, "
            "sin pronombres ambiguos. Responde SOLO con la pregunta.", temperatura=0).strip()

    def responder(self, pregunta):
        independiente = self._reformular(pregunta)                       # 1) reformular
        fragmentos = self.indice.buscar(independiente, self.k)           # 2) recuperar
        mensajes = [{"role": "user" if r == "Usuario" else "assistant", "content": t}
                    for r, t in self.historial[-self.memoria:]]          # 3) memoria como mensajes previos
        mensajes.append({"role": "user",
                         "content": f"FRAGMENTOS:\n{formatear_contexto(fragmentos)}\n\nPREGUNTA: {pregunta}"})
        respuesta = llm(mensajes, sistema=PROMPT_CURSO, temperatura=0.2).message.content   # 4) generar
        self.historial += [("Usuario", pregunta), ("Asistente", respuesta)]
        fuentes = sorted({f"{f['fuente']} pág. {f['pagina']}" for f in fragmentos})
        return f"{respuesta}\n\n*🔎 Fragmentos usados: {'; '.join(fuentes)}*"

bot_curso = ChatbotCurso(indice_curso)

# Demostración de memoria: la segunda pregunta solo tiene sentido gracias a la primera
display(Markdown("**❓ ¿Qué indica una vibración alta en una bomba?**\n\n" + bot_curso.responder("¿Qué indica una vibración alta en una bomba?")))
display(Markdown("**❓ ¿Y cómo se corrige?**\n\n" + bot_curso.responder("¿Y cómo se corrige?")))

In [ ]:
# ── 3.3 Chat interactivo con los documentos del curso ────────────────
crear_chat(bot_curso.responder, "🎓 Chatbot del curso")

<a name="4"></a>
# 4️⃣ Agente de mantenimiento

## 📘 Respuesta: diseño del agente

Un **agente** es un LLM que, en un **ciclo**, decide qué **herramienta** usar, recibe el resultado y continúa hasta resolver la tarea (*tool calling*, patrón ReAct):

```
 solicitud ─► LLM ─► ¿necesita herramienta? ──sí──► ejecuta función (Python) ─► resultado ─┐
                ▲                                                                          │
                └──────────────────────────── vuelve al LLM ◄──────────────────────────────┘
                                   no ─► respuesta final al usuario
```

**Las 4 herramientas pedidas** (cada una es una función de Python real sobre una base de datos SQLite):

| Herramienta | Qué hace |
|---|---|
| `buscar_repuesto(descripcion, equipo)` | Busca en el catálogo por descripción/síntoma/código |
| `consultar_almacen(codigo)` | Devuelve stock, mínimo, ubicación y plazo de reposición |
| `generar_orden_trabajo(equipo, falla, prioridad, repuestos)` | Crea la OT; reserva repuestos o la deja *pendiente de repuesto* |
| `actualizar_historial_fallas(equipo, falla, ot_id, causa_probable)` | Registra la falla y cuenta las anteriores del equipo |

**Buenas prácticas incluidas:** validación de entradas (el equipo debe existir), errores devueltos al LLM como JSON para que se corrija, límite de pasos del ciclo, **trazabilidad** (se imprime cada acción) y datos en BD (no en la memoria del modelo). En producción se añadiría **confirmación humana** antes de crear la OT.

In [ ]:
# ── 4.1 Base de datos del sistema de mantenimiento (SQLite) ──────────
import sqlite3, json, datetime, unicodedata
import pandas as pd

DB = "mantenimiento.db"

def conectar():
    return sqlite3.connect(DB)

def inicializar_bd():
    """Crea las tablas y las llena con datos de ejemplo (se reinicia cada vez que se ejecuta)."""
    con = conectar()
    con.executescript("""
    DROP TABLE IF EXISTS equipos;       DROP TABLE IF EXISTS repuestos;
    DROP TABLE IF EXISTS almacen;       DROP TABLE IF EXISTS ordenes_trabajo;
    DROP TABLE IF EXISTS historial_fallas;

    CREATE TABLE equipos(codigo TEXT PRIMARY KEY, descripcion TEXT, ubicacion TEXT, criticidad TEXT);
    CREATE TABLE repuestos(codigo TEXT PRIMARY KEY, descripcion TEXT, equipos_compatibles TEXT, palabras_clave TEXT);
    CREATE TABLE almacen(codigo TEXT PRIMARY KEY, stock INTEGER, minimo INTEGER, ubicacion TEXT, plazo_reposicion_dias INTEGER);
    CREATE TABLE ordenes_trabajo(id TEXT PRIMARY KEY, fecha TEXT, equipo TEXT, falla TEXT, prioridad TEXT, repuestos TEXT, estado TEXT);
    CREATE TABLE historial_fallas(id INTEGER PRIMARY KEY AUTOINCREMENT, fecha TEXT, equipo TEXT, falla TEXT, ot TEXT, causa_probable TEXT);
    """)
    con.executemany("INSERT INTO equipos VALUES (?,?,?,?)", [
        ("BOMBA-101", "Bomba centrífuga de agua de proceso", "Línea 1", "alta"),
        ("BOMBA-102", "Bomba centrífuga de reserva",        "Línea 1", "media"),
        ("MOTOR-201", "Motor eléctrico 30 kW",              "Línea 2", "alta"),
        ("COMPRESOR-301", "Compresor de aire de tornillo",  "Cuarto de máquinas", "media"),
    ])
    con.executemany("INSERT INTO repuestos VALUES (?,?,?,?)", [
        ("SM-BP101", "Sello mecánico 2 pulgadas para bomba centrífuga", "BOMBA-101;BOMBA-102", "sello fuga goteo empaque agua"),
        ("RD-6309",  "Rodamiento de bolas 6309-2RS",                    "MOTOR-201;BOMBA-101", "rodamiento ruido vibracion calentamiento temperatura"),
        ("AC-L100",  "Acoplamiento elástico L100",                      "BOMBA-101;MOTOR-201", "acoplamiento desalineacion vibracion"),
        ("IM-BP101", "Impulsor de bronce para bomba",                   "BOMBA-101;BOMBA-102", "impulsor cavitacion caudal bajo"),
        ("CR-B85",   "Correa trapezoidal B85",                          "COMPRESOR-301",       "correa patina ruido chirrido"),
        ("FA-C300",  "Filtro de aire del compresor",                    "COMPRESOR-301",       "filtro presion baja aire obstruido"),
        ("VT-301",   "Válvula de descarga del compresor",               "COMPRESOR-301",       "valvula fuga aire descarga"),
    ])
    con.executemany("INSERT INTO almacen VALUES (?,?,?,?,?)", [
        ("SM-BP101", 3, 2, "A-02-03", 7),
        ("RD-6309",  0, 4, "A-01-05", 15),   # SIN stock → para probar el caso de compra
        ("AC-L100",  1, 2, "A-03-01", 10),   # bajo el mínimo
        ("IM-BP101", 0, 1, "B-01-02", 30),
        ("CR-B85",   6, 3, "C-02-04", 5),
        ("FA-C300",  2, 2, "C-01-01", 5),
        ("VT-301",   1, 1, "C-03-02", 12),
    ])
    con.executemany("INSERT INTO historial_fallas(fecha,equipo,falla,ot,causa_probable) VALUES (?,?,?,?,?)", [
        ("2026-06-12", "BOMBA-101", "Vibración alta", "OT-2026-0001", "Desalineación del acoplamiento"),
        ("2026-08-03", "MOTOR-201", "Sobretemperatura", "OT-2026-0002", "Rodamiento lado ventilador"),
    ])
    con.commit(); con.close()

def tabla(nombre):
    """Muestra una tabla de la BD como DataFrame (para ver los cambios)."""
    con = conectar()
    df = pd.read_sql(f"SELECT * FROM {nombre}", con)
    con.close()
    return df

inicializar_bd()
display(tabla("almacen"))

In [ ]:
# ── 4.2 Las 4 herramientas (funciones de Python) ─────────────────────
# Cada herramienta devuelve un JSON (texto) para que el LLM lo lea fácilmente.

def norm(texto):
    """Minúsculas y sin acentos, para comparar textos de forma tolerante."""
    t = unicodedata.normalize("NFD", str(texto).lower())
    return "".join(c for c in t if unicodedata.category(c) != "Mn")

def clave(texto):
    """'Bomba 101' y 'BOMBA-101' → 'bomba101' (para comparar códigos de equipo)."""
    return re.sub(r"[^a-z0-9]", "", norm(texto))

PALABRAS_VACIAS = {"por", "para", "con", "del", "las", "los", "una", "uno", "que", "esta", "esto", "hay", "sin"}

# ───── Herramienta 1: buscar repuesto ─────
def buscar_repuesto(descripcion: str, equipo: str = None) -> str:
    con = conectar()
    filas = con.execute("SELECT codigo, descripcion, equipos_compatibles, palabras_clave FROM repuestos").fetchall()
    con.close()
    tokens = [t for t in re.findall(r"\w+", norm(descripcion)) if len(t) > 2 and t not in PALABRAS_VACIAS]

    def puntuar(filtrar_equipo):
        res = []
        for codigo, desc, equipos, claves in filas:
            if filtrar_equipo and equipo and clave(equipo) not in [clave(e) for e in equipos.split(";")]:
                continue                                    # el repuesto no es compatible con ese equipo
            texto = norm(f"{codigo} {desc} {claves}")
            puntaje = sum(1 for t in tokens if t in texto)  # cuántas palabras de la búsqueda aparecen
            if clave(descripcion) == clave(codigo):
                puntaje += 10                               # coincidencia exacta de código
            if puntaje > 0:
                res.append({"codigo": codigo, "descripcion": desc, "equipos_compatibles": equipos, "puntaje": puntaje})
        return sorted(res, key=lambda r: -r["puntaje"])[:3]

    resultados = puntuar(True) or puntuar(False)            # si no hay con ese equipo, amplía la búsqueda
    if not resultados:
        return json.dumps({"mensaje": "Sin coincidencias en el catálogo"}, ensure_ascii=False)
    return json.dumps({"resultados": resultados}, ensure_ascii=False)

# ───── Herramienta 2: consultar almacén ─────
def consultar_almacen(codigo: str) -> str:
    con = conectar()
    fila = con.execute("""SELECT a.codigo, r.descripcion, a.stock, a.minimo, a.ubicacion, a.plazo_reposicion_dias
                          FROM almacen a JOIN repuestos r ON r.codigo = a.codigo
                          WHERE upper(a.codigo) = upper(?)""", (codigo.strip(),)).fetchone()
    con.close()
    if not fila:
        return json.dumps({"error": f"El código {codigo} no existe en el almacén"}, ensure_ascii=False)
    cod, desc, stock, minimo, ubic, plazo = fila
    estado = "sin stock" if stock == 0 else ("bajo el mínimo" if stock < minimo else "disponible")
    return json.dumps({"codigo": cod, "descripcion": desc, "stock": stock, "minimo": minimo,
                       "estado": estado, "ubicacion": ubic, "plazo_reposicion_dias": plazo}, ensure_ascii=False)

# ───── Herramienta 3: generar orden de trabajo ─────
def generar_orden_trabajo(equipo: str, falla: str, prioridad: str, repuestos: list = None) -> str:
    con = conectar()
    equipos = {clave(c): c for (c,) in con.execute("SELECT codigo FROM equipos")}
    if clave(equipo) not in equipos:                        # validación: el equipo debe existir
        con.close()
        return json.dumps({"error": f"Equipo '{equipo}' no existe. Equipos válidos: {list(equipos.values())}"}, ensure_ascii=False)
    equipo = equipos[clave(equipo)]
    prioridad = norm(prioridad)
    if prioridad not in {"alta", "media", "baja"}:
        con.close()
        return json.dumps({"error": "prioridad debe ser alta, media o baja"}, ensure_ascii=False)

    reservados, faltantes = [], []
    for cod in (repuestos or []):                           # reservamos 1 unidad de cada repuesto con stock
        fila = con.execute("SELECT stock FROM almacen WHERE upper(codigo)=upper(?)", (cod,)).fetchone()
        if fila and fila[0] > 0:
            con.execute("UPDATE almacen SET stock = stock - 1 WHERE upper(codigo)=upper(?)", (cod,))
            reservados.append(cod)
        else:
            faltantes.append(cod)

    estado = "lista para ejecutar" if not faltantes else "pendiente de repuesto (solicitar compra)"
    n = con.execute("SELECT COUNT(*) FROM ordenes_trabajo").fetchone()[0] + 3   # +2 del historial de ejemplo, +1 siguiente
    ot_id = f"OT-{datetime.date.today().year}-{n:04d}"
    con.execute("INSERT INTO ordenes_trabajo VALUES (?,?,?,?,?,?,?)",
                (ot_id, datetime.date.today().isoformat(), equipo, falla, prioridad, ";".join(repuestos or []), estado))
    con.commit(); con.close()
    return json.dumps({"ot_id": ot_id, "equipo": equipo, "prioridad": prioridad, "estado": estado,
                       "repuestos_reservados": reservados, "repuestos_faltantes": faltantes}, ensure_ascii=False)

# ───── Herramienta 4: actualizar historial de fallas ─────
def actualizar_historial_fallas(equipo: str, falla: str, ot_id: str, causa_probable: str = "") -> str:
    con = conectar()
    equipos = {clave(c): c for (c,) in con.execute("SELECT codigo FROM equipos")}
    if clave(equipo) not in equipos:
        con.close()
        return json.dumps({"error": f"Equipo '{equipo}' no existe"}, ensure_ascii=False)
    equipo = equipos[clave(equipo)]
    previas = con.execute("SELECT COUNT(*) FROM historial_fallas WHERE equipo=?", (equipo,)).fetchone()[0]
    con.execute("INSERT INTO historial_fallas(fecha,equipo,falla,ot,causa_probable) VALUES (?,?,?,?,?)",
                (datetime.date.today().isoformat(), equipo, falla, ot_id, causa_probable))
    con.commit(); con.close()
    return json.dumps({"registrado": True, "equipo": equipo, "fallas_previas_del_equipo": previas,
                       "recurrente": previas >= 2}, ensure_ascii=False)

# Prueba manual de las herramientas, SIN el LLM (conviene probarlas antes de dárselas al agente)
print(buscar_repuesto("fuga por el sello", "BOMBA-101"))
print(consultar_almacen("SM-BP101"))

In [ ]:
# ── 4.3 Describir las herramientas al LLM (esquema JSON) ─────────────
# El LLM no ve el código Python: solo estas descripciones. Por eso deben ser claras.
HERRAMIENTAS = [
    {"type": "function", "function": {
        "name": "buscar_repuesto",
        "description": "Busca repuestos en el catálogo a partir de una descripción, síntoma o código. Devuelve hasta 3 candidatos con su código.",
        "parameters": {"type": "object", "properties": {
            "descripcion": {"type": "string", "description": "Texto de búsqueda, p. ej. 'sello fuga' o 'rodamiento ruido'"},
            "equipo": {"type": "string", "description": "Código del equipo, p. ej. BOMBA-101"}},
            "required": ["descripcion"]}}},
    {"type": "function", "function": {
        "name": "consultar_almacen",
        "description": "Consulta el stock, ubicación y plazo de reposición de un repuesto por su código exacto.",
        "parameters": {"type": "object", "properties": {
            "codigo": {"type": "string", "description": "Código del repuesto, p. ej. SM-BP101"}},
            "required": ["codigo"]}}},
    {"type": "function", "function": {
        "name": "generar_orden_trabajo",
        "description": "Crea una orden de trabajo (OT) para un equipo con falla. Reserva los repuestos con stock.",
        "parameters": {"type": "object", "properties": {
            "equipo": {"type": "string", "description": "Código del equipo"},
            "falla": {"type": "string", "description": "Descripción breve de la falla"},
            "prioridad": {"type": "string", "enum": ["alta", "media", "baja"]},
            "repuestos": {"type": "array", "items": {"type": "string"}, "description": "Códigos de repuestos requeridos"}},
            "required": ["equipo", "falla", "prioridad"]}}},
    {"type": "function", "function": {
        "name": "actualizar_historial_fallas",
        "description": "Registra la falla en el historial del equipo. Úsala DESPUÉS de crear la OT.",
        "parameters": {"type": "object", "properties": {
            "equipo": {"type": "string"},
            "falla": {"type": "string"},
            "ot_id": {"type": "string", "description": "ID de la OT creada, p. ej. OT-2026-0003"},
            "causa_probable": {"type": "string"}},
            "required": ["equipo", "falla", "ot_id"]}}},
]

# Diccionario nombre → función real, para ejecutar lo que el LLM pida
FUNCIONES = {f.__name__: f for f in [buscar_repuesto, consultar_almacen, generar_orden_trabajo, actualizar_historial_fallas]}

In [ ]:
# ── 4.4 El ciclo del agente ──────────────────────────────────────────
def construir_prompt_agente():
    equipos = "\n".join(f"- {c}: {d} ({u}, criticidad {k})" for c, d, u, k in conectar().execute("SELECT * FROM equipos"))
    return f"""Eres un agente de mantenimiento industrial. Gestionas fallas de equipos usando herramientas.

EQUIPOS VÁLIDOS:
{equipos}

FLUJO OBLIGATORIO (en este orden):
1. buscar_repuesto para identificar el/los repuestos necesarios.
2. consultar_almacen para CADA repuesto elegido.
3. generar_orden_trabajo (incluye los códigos de repuesto, también los que no tienen stock).
4. actualizar_historial_fallas con el ID de la OT devuelto.
5. Responde con un resumen breve: OT, repuestos y su disponibilidad, prioridad y siguiente paso.

REGLAS:
- Nunca inventes códigos ni IDs: usa solo lo que devuelven las herramientas.
- Prioridad ALTA si el equipo es de criticidad alta y hay riesgo de paro de línea o de seguridad, o si el repuesto no tiene stock.
- Si el repuesto no tiene stock, indícalo claramente y recomienda solicitar compra (menciona el plazo de reposición).
- Si el equipo no es identificable, pregunta al usuario en vez de adivinar.
- Responde en español."""

def agente_mantenimiento(solicitud, max_pasos=10, verbose=True):
    """Ciclo: LLM decide → ejecutamos herramienta → devolvemos resultado → repetimos."""
    mensajes = [{"role": "system", "content": construir_prompt_agente()},
                {"role": "user", "content": solicitud}]
    for paso in range(1, max_pasos + 1):
        r = ollama.chat(model=MODELO_LLM, messages=mensajes, tools=HERRAMIENTAS,
                        options={"temperature": 0, "num_ctx": 8192})
        msg = r.message
        mensajes.append(msg)                               # guardamos la decisión del LLM en el historial

        if not msg.tool_calls:                             # sin herramientas pedidas → respuesta final
            return msg.content

        for llamada in msg.tool_calls:                     # el LLM puede pedir varias a la vez
            nombre = llamada.function.name
            args = dict(llamada.function.arguments)
            if verbose:
                print(f"🔧 Paso {paso} → {nombre}({json.dumps(args, ensure_ascii=False)})")
            try:
                resultado = FUNCIONES[nombre](**args)      # ejecutamos la función real
            except Exception as e:                         # devolvemos el error al LLM para que se corrija
                resultado = json.dumps({"error": f"{type(e).__name__}: {e}"}, ensure_ascii=False)
            if verbose:
                print(f"   ↳ {resultado}")
            mensajes.append({"role": "tool", "tool_name": nombre, "content": resultado})
    return "⚠️ Se alcanzó el máximo de pasos sin terminar."

### 🎯 Escenario A — repuesto **con** stock
La bomba tiene una fuga por el sello. Esperamos: sello `SM-BP101` con stock → OT *lista para ejecutar* → historial actualizado.

In [ ]:
inicializar_bd()   # reinicia los datos para que el demo sea reproducible

respuesta = agente_mantenimiento(
    "La BOMBA-101 tiene una fuga de agua por el sello mecánico. Busca el repuesto, revisa el almacén, "
    "genera la orden de trabajo y actualiza el historial de fallas.")
print("\n" + "═" * 60 + "\n🤖 RESPUESTA FINAL\n" + "═" * 60)
display(Markdown(respuesta))

### 🎯 Escenario B — repuesto **sin** stock
El motor de la línea 2 hace ruido y se calienta. El rodamiento `RD-6309` tiene stock 0 → la OT debe quedar *pendiente de repuesto* y el agente debe recomendar compra.

In [ ]:
respuesta = agente_mantenimiento(
    "El MOTOR-201 hace ruido fuerte y se calienta, parece el rodamiento. Es urgente: la línea 2 depende de ese motor. "
    "Genera la orden de trabajo y actualiza el historial.")
print("\n" + "═" * 60 + "\n🤖 RESPUESTA FINAL\n" + "═" * 60)
display(Markdown(respuesta))

In [ ]:
# ── 4.5 Verificar qué cambió en la base de datos ─────────────────────
print("📋 Órdenes de trabajo"); display(tabla("ordenes_trabajo"))
print("📋 Historial de fallas"); display(tabla("historial_fallas"))
print("📦 Almacén (el stock de SM-BP101 bajó de 3 a 2 por la reserva)"); display(tabla("almacen"))

### ✅ Conclusión de la Sección 4
- El agente cumple los 4 requisitos: **busca repuesto → consulta almacén → genera OT (falla, equipo) → actualiza historial**.
- Las herramientas son **código determinista y probado**; el LLM solo decide *cuál* usar y con qué argumentos. Así se limita el riesgo de alucinaciones.
- **Mejoras para producción:** confirmación humana antes de crear la OT, autenticación y permisos por usuario, conexión a un CMMS/ERP real (SAP PM, Maximo), y una 5.ª herramienta `consultar_manual` que llame al RAG de la Sección 2 para sugerir el procedimiento de reparación.

<a name="5"></a>
# 5️⃣ ¿Podría un ESP32 usar un LLM? Restricciones

## 📘 Respuesta corta
**Sí, pero con límites muy fuertes.** Hay dos formas de entenderlo:

1. **LLM *dentro* del ESP32 (on-device):** posible solo con modelos **diminutos** (del orden de cientos de miles a pocos millones de parámetros, tipo *TinyStories*/`llama2.c`) en variantes con PSRAM como el ESP32-S3. Sirven como demostración o para tareas muy acotadas, **no** para razonar o responder preguntas técnicas generales.
2. **ESP32 como cliente de un LLM *externo*:** el ESP32 lee sensores, envía los datos por Wi-Fi/MQTT/HTTP a un servidor (por ejemplo **Ollama en la planta**) y recibe la respuesta. **Es la arquitectura realista** para aplicaciones industriales.

## ⚠️ Restricciones principales

| Restricción | ESP32 (valores típicos) | Consecuencia para un LLM |
|---|---|---|
| **RAM** | ~520 KB de SRAM; con PSRAM externa 2–8 MB (según módulo) | Un modelo de 7 B parámetros necesita **GB**; ni siquiera un modelo de 124 M cabe |
| **Almacenamiento (flash)** | 4–16 MB | Los pesos del modelo deben caber en flash o PSRAM |
| **Procesador** | 1–2 núcleos Xtensa/RISC-V a ~240 MHz, **sin GPU ni NPU** | Muy pocos tokens por segundo; la inferencia es multiplicar matrices |
| **Ancho de banda de memoria** | PSRAM por SPI (decenas de MB/s) | Cada token requiere leer **todos los pesos**: el límite no es solo el cálculo, es mover datos |
| **Contexto / caché KV** | Cada token de contexto consume memoria extra | Contextos muy cortos; sin RAG de manuales |
| **Precisión numérica** | FPU de precisión simple limitada | Se necesita **cuantización** (INT8/INT4), con pérdida de calidad |
| **Energía y calor** | Pensado para ahorro energético | Inferencia continua gasta batería y calienta |
| **Seguridad y conectividad** | TLS y pila Wi-Fi consumen RAM | Si se usa nube/servidor: latencia, credenciales y disponibilidad de red |
| **Determinismo y seguridad funcional** | Un LLM puede equivocarse | **Nunca** debe estar en el lazo de control de actuadores (paros, válvulas) |

## 🧭 Arquitecturas recomendadas
- **A. TinyML en el borde + LLM en servidor:** el ESP32 ejecuta un modelo pequeño (TensorFlow Lite Micro, Edge Impulse) para **detectar anomalías de vibración/temperatura**; solo cuando hay alarma envía un resumen al servidor, donde el LLM **explica la causa probable y consulta el RAG**.
- **B. ESP32 como terminal:** botón/pantalla/voz para que el operador reporte fallas al agente de la Sección 4 (el ESP32 solo captura y envía).
- **C. LLM diminuto local:** solo para demostraciones o comandos de vocabulario muy limitado.

## 🔢 Calculadora de viabilidad
La memoria necesaria ≈ **parámetros × bits por parámetro ÷ 8**. Además, como cada token obliga a leer todos los pesos, la velocidad máxima ≈ **ancho de banda de memoria ÷ tamaño del modelo**. La celda siguiente calcula ambas cosas (el ancho de banda es un **supuesto** de orden de magnitud que puedes cambiar).

In [ ]:
# ── 5.1 ¿Cabe un modelo en un ESP32? ─────────────────────────────────
import matplotlib.pyplot as plt

# Memorias típicas (MB). Cambia estos valores según tu módulo ESP32.
MEMORIA_MB = {"SRAM (520 KB)": 0.52, "PSRAM (8 MB)": 8, "Flash (16 MB)": 16}
ANCHO_BANDA_MBPS = 40        # SUPUESTO: ancho de banda efectivo de PSRAM en MB/s (orden de magnitud)

# Modelos de ejemplo: nombre → número de parámetros
MODELOS = {
    "TinyStories 260K": 0.26e6,
    "TinyStories 15M":  15e6,
    "GPT-2 small 124M": 124e6,
    "TinyLlama 1.1B":   1.1e9,
    "Qwen2.5 7B (el de este notebook)": 7e9,
}
BITS = {"FP16": 16, "INT8": 8, "INT4": 4}   # niveles de cuantización

filas = []
for modelo, params in MODELOS.items():
    for etiqueta, bits in BITS.items():
        mb = params * bits / 8 / 1e6                 # tamaño en MB (sin contar caché KV ni activaciones)
        filas.append({
            "Modelo": modelo, "Cuantización": etiqueta, "Tamaño (MB)": round(mb, 2),
            "¿Cabe en PSRAM 8 MB?": "✅" if mb <= MEMORIA_MB["PSRAM (8 MB)"] * 0.8 else "❌",   # dejamos 20 % para el resto
            "¿Cabe en flash 16 MB?": "✅" if mb <= MEMORIA_MB["Flash (16 MB)"] * 0.8 else "❌",
            "Tokens/s máx. teórico": round(ANCHO_BANDA_MBPS / mb, 2),
        })
df_esp = pd.DataFrame(filas)
display(df_esp)

# Gráfico: tamaño de cada modelo vs. memoria disponible (escala logarítmica)
fig, ax = plt.subplots(figsize=(9, 4.5))
for modelo in MODELOS:
    sub = df_esp[df_esp["Modelo"] == modelo]
    ax.plot(sub["Cuantización"], sub["Tamaño (MB)"], marker="o", label=modelo)
for nombre, mb in MEMORIA_MB.items():
    ax.axhline(mb, linestyle="--", linewidth=1, color="gray")
    ax.text(2.02, mb, nombre, va="center", fontsize=8, color="gray")
ax.set_yscale("log")
ax.set_ylabel("Tamaño del modelo (MB, escala log)")
ax.set_title("Tamaño de modelos cuantizados vs. memoria típica del ESP32")
ax.legend(fontsize=7, loc="upper right")
plt.tight_layout()
plt.show()

### 📈 Lectura de resultados
- Con 8 MB de PSRAM, solo un modelo de **~260 K parámetros** cabe con holgura. Uno de 15 M apenas entraría en INT4 (~7.5 MB) y sin dejar espacio para la caché KV; por encima de eso (GPT-2 de 124 M, TinyLlama, Qwen) ya no hay opción. Un modelo de 7 B necesitaría **≈3.5 GB incluso en INT4**: unas 400 veces la PSRAM de 8 MB.
- La columna "Tokens/s máx. teórico" es una **cota superior**: en la práctica el cálculo en un microcontrolador sin GPU lo reduce aún más.
- Por eso, en aplicaciones de mantenimiento, el ESP32 se usa como **sensor/terminal inteligente** y el LLM vive en un servidor (arquitectura A o B).

### 🔌 Ejemplo de arquitectura B: ESP32 → servidor con Ollama (MicroPython, referencia)
Este código **no se ejecuta en Colab**: es un ejemplo de lo que correría en el ESP32. Envía una lectura de sensores al LLM del servidor de la planta (cambia `192.168.1.50` por la IP del equipo donde corre Ollama, con `OLLAMA_HOST=0.0.0.0`).

```python
import urequests, ujson, network

# 1) Conectar a Wi-Fi (se omite el código de conexión)
SERVIDOR = "http://192.168.1.50:11434/api/generate"

def consultar_llm(vibracion_mm_s, temperatura_c):
    prompt = ("Eres un técnico de mantenimiento. Una bomba tiene vibración "
              f"{vibracion_mm_s} mm/s y temperatura {temperatura_c} °C. "
              "Indica en 2 frases si es normal y qué revisar.")
    cuerpo = ujson.dumps({"model": "qwen2.5:7b", "prompt": prompt, "stream": False})
    r = urequests.post(SERVIDOR, data=cuerpo, headers={"Content-Type": "application/json"})
    texto = r.json()["response"]      # respuesta del LLM
    r.close()
    return texto

print(consultar_llm(8.4, 78))
```

### ✅ Conclusión de la Sección 5
Un ESP32 **puede** ejecutar un LLM *de juguete* o **conectarse** a un LLM real, pero **no** ejecutar uno útil por sí solo. Las restricciones decisivas son **memoria (SRAM/PSRAM), ancho de banda de memoria, potencia de cálculo y energía**; además, por seguridad, el LLM debe **asesorar**, no controlar actuadores.

# 📦 Entregables y cierre
La celda siguiente empaqueta todo lo generado (skills, HTML, código con pruebas, base de datos) en `entrega.zip` y lo descarga.

| Punto | Entregable |
|---|---|
| 1 | Carpeta `skills/` con los 4 `SKILL.md` + `explicacion.html` |
| 2 | Índice RAG y chatbot de manuales (Sección 2) |
| 3 | Chatbot conversacional con documentos del curso (Sección 3) |
| 4 | `mantenimiento.db` y agente con 4 herramientas (Sección 4) |
| 5 | Análisis y calculadora ESP32 (Sección 5) |

In [ ]:
# ── Empaquetar los resultados en un ZIP ──────────────────────────────
import zipfile

with zipfile.ZipFile("entrega.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for ruta in ["skills", "trabajo_impl", "explicacion.html", "mantenimiento.db", "especificacion_ejemplo.txt"]:
        p = Path(ruta)
        if p.is_dir():
            for f in p.rglob("*"):
                if f.is_file() and "__pycache__" not in str(f):
                    z.write(f)
        elif p.exists():
            z.write(p)
print("📦 entrega.zip creado con:", zipfile.ZipFile("entrega.zip").namelist())

try:                                    # la descarga solo funciona dentro de Colab
    from google.colab import files
    files.download("entrega.zip")
except ImportError:
    pass